## CloudWatch Metrics and Dashboards

Welcome to the next step in your AWS developer journey! So far, you have learned how to keep your applications secure by managing credentials and protecting your APIs. Now, we will focus on another critical aspect: monitoring your applications.

Monitoring is essential for both security and reliability. It helps you answer questions like:

* Is my application working as expected?
* Are there any unusual patterns or errors?
* How is my application performing over time?

AWS CloudWatch is a service that helps you collect, view, and analyze data from your applications. In this lesson, you will learn how to send your own custom data (called metrics) to CloudWatch and create dashboards to visualize this information. By the end, you will be able to track important numbers from your application and see them in real time.

---

## Quick Recall: Using `boto3` with AWS Services

Before we dive in, let's quickly remind ourselves about boto3. In previous lessons, you used boto3 to interact with AWS services like Secrets Manager and Lambda. boto3 is the official AWS SDK for Python, and it allows you to connect to AWS services directly from your code.

For example, to use a service, you typically create a client like this:

```python
import boto3

# Create a client for AWS CloudWatch
cw = boto3.client('cloudwatch')
```

This `cw` object lets you call CloudWatch functions from your Python code. You will use this same approach to send metrics and create dashboards in this lesson.

---

## Understanding the CloudWatch Flow

Before we start coding, let's visualize how the pieces fit together:

```text
┌─────────────────────┐
│  Python Application │
│   (Your Code)       │
└──────────┬──────────┘
           │
           │ boto3.client('cloudwatch')
           │ put_metric_data()
           ↓
┌─────────────────────┐
│   AWS CloudWatch    │
│  (Stores Metrics)   │
└──────────┬──────────┘
           │
           │ Visualizes
           ↓
┌─────────────────────┐
│  CloudWatch         │
│  Dashboard          │
└─────────────────────┘
```

This diagram shows the complete flow:

1. Your Python application sends metrics to CloudWatch using boto3
2. CloudWatch receives and stores these metrics
3. A dashboard displays the metrics visually for monitoring

> **Important note about timing:** When you send metrics to CloudWatch, they don't appear instantly. It can take several minutes (typically 2-5 minutes) for your metrics to show up in the CloudWatch console or on dashboards. This is normal behavior—CloudWatch processes and aggregates data in the background. So if you don't see your metrics right away, be patient and refresh the dashboard after a few minutes.

Now let's implement this flow step by step.

---

## Emitting Custom Metrics to CloudWatch

A metric is just a number that tells you something about your application. For example, you might want to track the value of each order placed in your app.

Let's see how to send a custom metric to CloudWatch step by step.

### Step 1: Import Required Libraries

First, you need to import the libraries you will use. You will need `boto3` for AWS and some standard libraries for working with time and random numbers.

```python
import time
import random
import boto3
```

* `time` helps you add timestamps to your metrics.
* `random` is used here to generate example data.
* `boto3` is for connecting to AWS CloudWatch.

### Step 2: Create a CloudWatch Client

Next, create a CloudWatch client using boto3:

```python
cw = boto3.client('cloudwatch')
```

This line sets up the connection to CloudWatch.

### Step 3: Define a Function to Send a Metric

Now, let's write a function that sends a single metric value to CloudWatch. We will call this metric `OrderValue`.

```python
def put_metric_value(value):
    """Emit a single metric data point to CloudWatch"""
    cw.put_metric_data(
        Namespace="App/Metrics",
        MetricData=[{
            "MetricName": "OrderValue",
            "Dimensions": [{"Name":"Service","Value":"OrdersAPI"}],
            "Timestamp": time.time(),
            "Value": value
        }]
    )
```

Let's break down what's happening here:

* `Namespace` is a way to group related metrics. Here, we use `"App/Metrics"`.
* `MetricName` is the name of the metric, in this case, `"OrderValue"`.
* `Dimensions` are extra labels to help you filter or group your data. Here, we label the metric with `"Service": "OrdersAPI"`.
* `Timestamp` records when the metric was sent.
* `Value` is the actual number you want to track.

### Step 4: Emit Some Example Metrics

Let's send a few example metrics to CloudWatch. We'll use random numbers to simulate order values.

```python
for i in range(5):
    value = random.uniform(10, 200)
    put_metric_value(value)
    print(f"Emitted metric: OrderValue = {value:.2f}")
    time.sleep(1)
```

This code will:

1. Generate a random order value between 10 and 200.
2. Send it to CloudWatch using your function.
3. Print out what was sent.
4. Wait one second before sending the next value.

**Example output:**

```text
Emitted metric: OrderValue = 153.27
Emitted metric: OrderValue = 45.12
Emitted metric: OrderValue = 189.03
Emitted metric: OrderValue = 77.56
Emitted metric: OrderValue = 120.44
```

Now, you have sent custom metrics to CloudWatch!

---

## Creating and Configuring a CloudWatch Dashboard

A dashboard in CloudWatch is a visual display of your metrics. It helps you see trends and spot problems quickly.

Let's see how to create a dashboard step by step.

### Step 1: Import the JSON Library

You will need the `json` library to build the dashboard configuration.

```python
import json
```

### Step 2: Define a Function to Create a Dashboard

Now, let's write a function that creates a dashboard and adds a widget to show your `OrderValue` metric.

```python
def create_dashboard():
    """Create CloudWatch dashboard to visualize metrics"""
    widget = {
        "type":"metric","x":0,"y":0,"width":12,"height":6,
        "properties":{
            "metrics":[["App/Metrics","OrderValue","Service","OrdersAPI"]],
            "stat":"Average","title":"Order Value",
            "region":"us-east-1",  # Add your AWS region here
            "annotations":{"horizontal":[]},
            "period":300
        }
    }
    cw.put_dashboard(
        DashboardName="App-Overview", 
        DashboardBody=json.dumps({"widgets":[widget]})
    )
```

Here's what's happening:

* We define a `widget` that tells CloudWatch what to display.
* The widget shows the average value of the `OrderValue` metric for the `OrdersAPI` service.
* The dashboard is named `"App-Overview"`.
* The dashboard is created or updated using `cw.put_dashboard`.

### Step 3: Create the Dashboard

Now, call the function to create the dashboard:

```python
create_dashboard()
print("Dashboard 'App-Overview' created successfully!")
```

**Example output:**

```text
Dashboard 'App-Overview' created successfully!
```

You can now go to the AWS CloudWatch console and see your dashboard with the `OrderValue` metric displayed.

---

## Summary and Practice Preview

In this lesson, you learned how to:

* Send custom metrics from your application to AWS CloudWatch using Python and boto3.
* Create a CloudWatch dashboard to visualize your metrics in real time.

These skills help you keep an eye on your application's health and performance, which is important for both security and reliability.

Next, you will get hands-on practice by emitting your own metrics and building dashboards in the CodeSignal environment. Remember, on CodeSignal, the required libraries are already installed, so you can focus on writing and running your code.

Great job making it this far! You are now ready to monitor your own AWS applications and gain valuable insights from your data.

## Complete Your First CloudWatch Metric

Now that you understand how CloudWatch metrics work, it's time to put your knowledge into practice! You have been given a partially working `put_metric_value` function that already has the correct namespace, metric name, and dimensions set up. However, two important pieces are missing from the metric data.

Your objective is to complete the function by adding the missing fields:

* Add the `timestamp` field to record when the metric was sent.
* Add the `value` field to include the actual metric data.

Look for the TODO comments in the `put_metric_value` function — they will guide you to exactly where you need to add the missing code. The timestamp should use `time.time()` to get the current time, and the value should use the `value` parameter that is passed to the function.

Once you complete the function, the code will emit five sample metrics to CloudWatch and create a dashboard to visualize them. This hands-on practice will help you master the essential skill of sending custom metrics to monitor your applications effectively.

```python
import time, random, json, boto3

# Initialize CloudWatch client
cw = boto3.client('cloudwatch')
NAMESPACE = "App/Metrics"
DASH_NAME = "App-Overview"

def put_metric_value(value):
    """Emit a single metric data point to CloudWatch"""
    cw.put_metric_data(
        Namespace=NAMESPACE,
        MetricData=[{
            "MetricName": "OrderValue",
            "Dimensions": [{"Name":"Service","Value":"OrdersAPI"}],
            # TODO: Add the Timestamp field using time.time()
            # TODO: Add the Value field using the value parameter
        }]
    )

def create_dashboard():
    """Create CloudWatch dashboard to visualize metrics"""
    widget = {
        "type":"metric","x":0,"y":0,"width":12,"height":6,
        "properties":{
            "metrics":[["App/Metrics","OrderValue","Service","OrdersAPI"]],
            "stat":"Average","title":"Order Value",
            "region":"us-east-1",  # Add your AWS region here
            "annotations":{"horizontal":[]},
            "period":300
        }
    }
    cw.put_dashboard(
        DashboardName=DASH_NAME, 
        DashboardBody=json.dumps({"widgets":[widget]})
    )

if __name__ == "__main__":
    print("Starting application monitoring setup...")
    
    # Step 1: Emit sample metrics
    print("Emitting sample metrics...")
    for i in range(5):
        value = random.uniform(10, 200)
        put_metric_value(value)
        print(f"Emitted metric: OrderValue = {value:.2f}")
        time.sleep(1)
    
    # Step 2: Create dashboard
    print("Creating CloudWatch dashboard...")
    create_dashboard()
    print(f"Dashboard '{DASH_NAME}' created successfully!")
    
    print("Setup complete. Check AWS CloudWatch console to view your metrics and dashboard.")
```

Here is the completed `put_metric_value` function with `Timestamp` and `Value` added:

```python
import time, random, json, boto3

# Initialize CloudWatch client
cw = boto3.client('cloudwatch')
NAMESPACE = "App/Metrics"
DASH_NAME = "App-Overview"

def put_metric_value(value):
    """Emit a single metric data point to CloudWatch"""
    cw.put_metric_data(
        Namespace=NAMESPACE,
        MetricData=[{
            "MetricName": "OrderValue",
            "Dimensions": [{"Name":"Service","Value":"OrdersAPI"}],
            "Timestamp": time.time(),
            "Value": value
        }]
    )

def create_dashboard():
    """Create CloudWatch dashboard to visualize metrics"""
    widget = {
        "type":"metric","x":0,"y":0,"width":12,"height":6,
        "properties":{
            "metrics":[["App/Metrics","OrderValue","Service","OrdersAPI"]],
            "stat":"Average","title":"Order Value",
            "region":"us-east-1",  # Add your AWS region here
            "annotations":{"horizontal":[]},
            "period":300
        }
    }
    cw.put_dashboard(
        DashboardName=DASH_NAME, 
        DashboardBody=json.dumps({"widgets":[widget]})
    )

if __name__ == "__main__":
    print("Starting application monitoring setup...")
    
    # Step 1: Emit sample metrics
    print("Emitting sample metrics...")
    for i in range(5):
        value = random.uniform(10, 200)
        put_metric_value(value)
        print(f"Emitted metric: OrderValue = {value:.2f}")
        time.sleep(1)
    
    # Step 2: Create dashboard
    print("Creating CloudWatch dashboard...")
    create_dashboard()
    print(f"Dashboard '{DASH_NAME}' created successfully!")
    
    print("Setup complete. Check AWS CloudWatch console to view your metrics and dashboard.")
```

## Adding Metric Dimensions for Better Organization

Excellent work on sending your first custom metrics to CloudWatch! Now it's time to learn about dimensions — a powerful feature that helps you organize and filter your metrics.

Currently, your `put_metric_value` function sends metrics with only one dimension: `"Service"` set to `"OrdersAPI"`. While this works, you can make your metrics much more useful by adding additional dimensions. Think of dimensions as labels that help you group and filter your data in CloudWatch.

Your task is to add a second dimension to make your metrics more specific. You need to modify the `put_metric_value` function to include a `"Region"` dimension with the value `"us-east-1"`.

Look for the TODO comment in the function — it will show you exactly where to add the new dimension. Remember that dimensions are stored in a list, so you'll need to add another dictionary with `"Name"` and `"Value"` keys, just like the existing `"Service"` dimension.

This exercise will teach you how to organize metrics by multiple criteria, which is essential when monitoring applications across different regions and services.

```python
import time, random, json, boto3

# Initialize CloudWatch client
cw = boto3.client('cloudwatch')
NAMESPACE = "App/Metrics"
DASH_NAME = "App-Overview"

def put_metric_value(value):
    """Emit a single metric data point to CloudWatch"""
    cw.put_metric_data(
        Namespace=NAMESPACE,
        MetricData=[{
            "MetricName": "OrderValue",
            # TODO: Add a second dimension with Name="Region" and Value="us-east-1"
            "Dimensions": [
                {"Name":"Service","Value":"OrdersAPI"}
            ],
            "Timestamp": time.time(),
            "Value": value
        }]
    )

def create_dashboard():
    """Create CloudWatch dashboard to visualize metrics"""
    widget = {
        "type":"metric","x":0,"y":0,"width":12,"height":6,
        "properties":{
            "metrics":[["App/Metrics","OrderValue","Service","OrdersAPI"]],
            "stat":"Average","title":"Order Value",
            "region":"us-east-1",  # Add your AWS region here
            "annotations":{"horizontal":[]},
            "period":300
        }
    }
    cw.put_dashboard(
        DashboardName=DASH_NAME, 
        DashboardBody=json.dumps({"widgets":[widget]})
    )

if __name__ == "__main__":
    print("Starting application monitoring setup...")
    
    # Step 1: Emit sample metrics
    print("Emitting sample metrics...")
    for i in range(5):
        value = random.uniform(10, 200)
        put_metric_value(value)
        print(f"Emitted metric: OrderValue = {value:.2f}")
        time.sleep(1)
    
    # Step 2: Create dashboard
    print("Creating CloudWatch dashboard...")
    create_dashboard()
    print(f"Dashboard '{DASH_NAME}' created successfully!")
    
    print("Setup complete. Check AWS CloudWatch console to view your metrics and dashboard.")
```

Here is the completed `put_metric_value` function with the `"Region"` dimension added:

```python
import time, random, json, boto3

# Initialize CloudWatch client
cw = boto3.client('cloudwatch')
NAMESPACE = "App/Metrics"
DASH_NAME = "App-Overview"

def put_metric_value(value):
    """Emit a single metric data point to CloudWatch"""
    cw.put_metric_data(
        Namespace=NAMESPACE,
        MetricData=[{
            "MetricName": "OrderValue",
            "Dimensions": [
                {"Name":"Service","Value":"OrdersAPI"},
                {"Name":"Region","Value":"us-east-1"}
            ],
            "Timestamp": time.time(),
            "Value": value
        }]
    )

def create_dashboard():
    """Create CloudWatch dashboard to visualize metrics"""
    widget = {
        "type":"metric","x":0,"y":0,"width":12,"height":6,
        "properties":{
            "metrics":[["App/Metrics","OrderValue","Service","OrdersAPI"]],
            "stat":"Average","title":"Order Value",
            "region":"us-east-1",  # Add your AWS region here
            "annotations":{"horizontal":[]},
            "period":300
        }
    }
    cw.put_dashboard(
        DashboardName=DASH_NAME, 
        DashboardBody=json.dumps({"widgets":[widget]})
    )

if __name__ == "__main__":
    print("Starting application monitoring setup...")
    
    # Step 1: Emit sample metrics
    print("Emitting sample metrics...")
    for i in range(5):
        value = random.uniform(10, 200)
        put_metric_value(value)
        print(f"Emitted metric: OrderValue = {value:.2f}")
        time.sleep(1)
    
    # Step 2: Create dashboard
    print("Creating CloudWatch dashboard...")
    create_dashboard()
    print(f"Dashboard '{DASH_NAME}' created successfully!")
    
    print("Setup complete. Check AWS CloudWatch console to view your metrics and dashboard.")
```

## Batching Multiple Metrics Efficiently

Nice work on organizing your metrics with dimensions! Now you're ready to learn a more efficient approach: batching multiple metrics in a single API call.

Currently, if you want to send three different metrics (like order value, order count, and processing time), you need to make three separate calls to CloudWatch. This works, but it's not the most efficient way. CloudWatch allows you to send multiple metrics at once using the `MetricData` list in a single `put_metric_data` call.

Your task is to complete the `put_multiple_metrics` function that sends three different metrics in one API call:

* `OrderValue` — the dollar amount of an order
* `OrderCount` — how many orders were processed
* `ProcessingTime` — how long it took to process the orders (in milliseconds)

The starter code already includes the first metric (`OrderValue`) as a complete example. You need to add the other two metrics to the `MetricData` list. Look for the TODO comments that show you exactly where to add the missing metrics.

Each new metric should follow the same structure as the first one: use the appropriate parameter, include the same dimensions, and use the same timestamp. This exercise will teach you how to efficiently batch metrics and better understand the `MetricData` structure.

```python
import time, random, json, boto3

# Initialize CloudWatch client
cw = boto3.client('cloudwatch')
NAMESPACE = "App/Metrics"
DASH_NAME = "App-Overview"

def put_multiple_metrics(order_value, order_count, processing_time):
    """Emit multiple metric data points to CloudWatch in a single call"""
    cw.put_metric_data(
        Namespace=NAMESPACE,
        MetricData=[
            {
                "MetricName": "OrderValue",
                "Dimensions": [
                    {"Name":"Service","Value":"OrdersAPI"},
                    {"Name":"Region","Value":"us-east-1"}
                ],
                "Timestamp": time.time(),
                "Value": order_value
            },
            # TODO: Add OrderCount metric using the order_count parameter
            # TODO: Add ProcessingTime metric using the processing_time parameter
        ]
    )

def create_dashboard():
    """Create CloudWatch dashboard to visualize metrics"""
    widgets = [
        {
            "type":"metric","x":0,"y":0,"width":8,"height":6,
            "properties":{
                "metrics":[["App/Metrics","OrderValue","Service","OrdersAPI"]],
                "stat":"Average","title":"Order Value",
                "region":"us-east-1",
                "annotations":{"horizontal":[]},
                "period":300
            }
        },
        {
            "type":"metric","x":8,"y":0,"width":8,"height":6,
            "properties":{
                "metrics":[["App/Metrics","OrderCount","Service","OrdersAPI"]],
                "stat":"Sum","title":"Order Count",
                "region":"us-east-1",
                "annotations":{"horizontal":[]},
                "period":300
            }
        },
        {
            "type":"metric","x":16,"y":0,"width":8,"height":6,
            "properties":{
                "metrics":[["App/Metrics","ProcessingTime","Service","OrdersAPI"]],
                "stat":"Average","title":"Processing Time (ms)",
                "region":"us-east-1",
                "annotations":{"horizontal":[]},
                "period":300
            }
        }
    ]
    
    cw.put_dashboard(
        DashboardName=DASH_NAME, 
        DashboardBody=json.dumps({"widgets":widgets})
    )

if __name__ == "__main__":
    print("Starting application monitoring setup...")
    
    # Step 1: Emit sample metrics
    print("Emitting sample metrics...")
    for i in range(5):
        order_value = random.uniform(10, 200)
        order_count = random.randint(1, 10)
        processing_time = random.uniform(50, 500)
        
        put_multiple_metrics(order_value, order_count, processing_time)
        print(f"Emitted metrics: OrderValue={order_value:.2f}, OrderCount={order_count}, ProcessingTime={processing_time:.2f}ms")
        time.sleep(1)
    
    # Step 2: Create dashboard
    print("Creating CloudWatch dashboard...")
    create_dashboard()
    print(f"Dashboard '{DASH_NAME}' created successfully!")
    
    print("Setup complete. Check AWS CloudWatch console to view your metrics and dashboard.")
```

Here is the completed `put_multiple_metrics` function with `OrderCount` and `ProcessingTime` added. A single `ts` is captured once and reused across all three entries so they share the exact same timestamp:

```python
import time, random, json, boto3

# Initialize CloudWatch client
cw = boto3.client('cloudwatch')
NAMESPACE = "App/Metrics"
DASH_NAME = "App-Overview"

def put_multiple_metrics(order_value, order_count, processing_time):
    """Emit multiple metric data points to CloudWatch in a single call"""
    ts = time.time()
    cw.put_metric_data(
        Namespace=NAMESPACE,
        MetricData=[
            {
                "MetricName": "OrderValue",
                "Dimensions": [
                    {"Name":"Service","Value":"OrdersAPI"},
                    {"Name":"Region","Value":"us-east-1"}
                ],
                "Timestamp": ts,
                "Value": order_value
            },
            {
                "MetricName": "OrderCount",
                "Dimensions": [
                    {"Name":"Service","Value":"OrdersAPI"},
                    {"Name":"Region","Value":"us-east-1"}
                ],
                "Timestamp": ts,
                "Value": order_count
            },
            {
                "MetricName": "ProcessingTime",
                "Dimensions": [
                    {"Name":"Service","Value":"OrdersAPI"},
                    {"Name":"Region","Value":"us-east-1"}
                ],
                "Timestamp": ts,
                "Value": processing_time
            }
        ]
    )

def create_dashboard():
    """Create CloudWatch dashboard to visualize metrics"""
    widgets = [
        {
            "type":"metric","x":0,"y":0,"width":8,"height":6,
            "properties":{
                "metrics":[["App/Metrics","OrderValue","Service","OrdersAPI"]],
                "stat":"Average","title":"Order Value",
                "region":"us-east-1",
                "annotations":{"horizontal":[]},
                "period":300
            }
        },
        {
            "type":"metric","x":8,"y":0,"width":8,"height":6,
            "properties":{
                "metrics":[["App/Metrics","OrderCount","Service","OrdersAPI"]],
                "stat":"Sum","title":"Order Count",
                "region":"us-east-1",
                "annotations":{"horizontal":[]},
                "period":300
            }
        },
        {
            "type":"metric","x":16,"y":0,"width":8,"height":6,
            "properties":{
                "metrics":[["App/Metrics","ProcessingTime","Service","OrdersAPI"]],
                "stat":"Average","title":"Processing Time (ms)",
                "region":"us-east-1",
                "annotations":{"horizontal":[]},
                "period":300
            }
        }
    ]
    
    cw.put_dashboard(
        DashboardName=DASH_NAME, 
        DashboardBody=json.dumps({"widgets":widgets})
    )

if __name__ == "__main__":
    print("Starting application monitoring setup...")
    
    # Step 1: Emit sample metrics
    print("Emitting sample metrics...")
    for i in range(5):
        order_value = random.uniform(10, 200)
        order_count = random.randint(1, 10)
        processing_time = random.uniform(50, 500)
        
        put_multiple_metrics(order_value, order_count, processing_time)
        print(f"Emitted metrics: OrderValue={order_value:.2f}, OrderCount={order_count}, ProcessingTime={processing_time:.2f}ms")
        time.sleep(1)
    
    # Step 2: Create dashboard
    print("Creating CloudWatch dashboard...")
    create_dashboard()
    print(f"Dashboard '{DASH_NAME}' created successfully!")
    
    print("Setup complete. Check AWS CloudWatch console to view your metrics and dashboard.")
```

## Fix Broken Dashboard Widget Configuration

Perfect! You've mastered sending metrics efficiently with batching. Now, let's tackle a common challenge that occurs when creating dashboards: configuration errors that prevent your metrics from displaying properly.

You have a monitoring setup that successfully sends metrics to CloudWatch, but there's a problem with the dashboard. When you check the AWS console, the dashboard exists but shows no data because of configuration issues in the widget setup.

Your task is to fix two specific problems in the `create_dashboard` function:

* Fix the `metrics` array format to match CloudWatch's required nested list structure
* Add the missing `region` property that CloudWatch needs to display the data

Look for the TODO comments that will guide you to the exact locations where these fixes are needed. Once you correct these configuration issues, your dashboard will properly visualize the `OrderValue` metrics and give you a working monitoring solution.

```python
import time, random, json, boto3

# Initialize CloudWatch client
cw = boto3.client('cloudwatch')
NAMESPACE = "App/Metrics"
DASH_NAME = "App-Overview"

def create_dashboard():
    """Create CloudWatch dashboard to visualize metrics"""
    widget = {
        "type":"metric","x":0,"y":0,"width":12,"height":6,
        "properties":{
            # TODO: Fix the metrics array format - it should be a nested list structure
            "metrics":["App/Metrics","OrderValue","Service","OrdersAPI"],
            "stat":"Average","title":"Order Value",
            # TODO: Add the missing region property set to "us-east-1"
            "annotations":{"horizontal":[]},
            "period":300
        }
    }
    cw.put_dashboard(
        DashboardName=DASH_NAME, 
        DashboardBody=json.dumps({"widgets":[widget]})
    )

def put_metric_value(value):
    """Emit a single metric data point to CloudWatch"""
    cw.put_metric_data(
        Namespace=NAMESPACE,
        MetricData=[{
            "MetricName": "OrderValue",
            "Dimensions": [{"Name":"Service","Value":"OrdersAPI"}],
            "Timestamp": time.time(),
            "Value": value
        }]
    )

if __name__ == "__main__":
    print("Starting application monitoring setup...")
    
    # Step 1: Emit sample metrics
    print("Emitting sample metrics...")
    for i in range(5):
        value = random.uniform(10, 200)
        put_metric_value(value)
        print(f"Emitted metric: OrderValue = {value:.2f}")
        time.sleep(1)
    
    # Step 2: Create dashboard
    print("Creating CloudWatch dashboard...")
    create_dashboard()
    print(f"Dashboard '{DASH_NAME}' created successfully!")
    
    print("Setup complete. Check AWS CloudWatch console to view your metrics and dashboard.")
```

Here is the completed `create_dashboard` function with both fixes applied — `metrics` wrapped in a nested list (`[[...]]`, matching the `[["App/Metrics","OrderValue","Service","OrdersAPI"]]` pattern seen throughout the lesson) and `region` added:

```python
import time, random, json, boto3

# Initialize CloudWatch client
cw = boto3.client('cloudwatch')
NAMESPACE = "App/Metrics"
DASH_NAME = "App-Overview"

def create_dashboard():
    """Create CloudWatch dashboard to visualize metrics"""
    widget = {
        "type":"metric","x":0,"y":0,"width":12,"height":6,
        "properties":{
            "metrics":[["App/Metrics","OrderValue","Service","OrdersAPI"]],
            "stat":"Average","title":"Order Value",
            "region":"us-east-1",
            "annotations":{"horizontal":[]},
            "period":300
        }
    }
    cw.put_dashboard(
        DashboardName=DASH_NAME, 
        DashboardBody=json.dumps({"widgets":[widget]})
    )

def put_metric_value(value):
    """Emit a single metric data point to CloudWatch"""
    cw.put_metric_data(
        Namespace=NAMESPACE,
        MetricData=[{
            "MetricName": "OrderValue",
            "Dimensions": [{"Name":"Service","Value":"OrdersAPI"}],
            "Timestamp": time.time(),
            "Value": value
        }]
    )

if __name__ == "__main__":
    print("Starting application monitoring setup...")
    
    # Step 1: Emit sample metrics
    print("Emitting sample metrics...")
    for i in range(5):
        value = random.uniform(10, 200)
        put_metric_value(value)
        print(f"Emitted metric: OrderValue = {value:.2f}")
        time.sleep(1)
    
    # Step 2: Create dashboard
    print("Creating CloudWatch dashboard...")
    create_dashboard()
    print(f"Dashboard '{DASH_NAME}' created successfully!")
    
    print("Setup complete. Check AWS CloudWatch console to view your metrics and dashboard.")
```

`metrics` was a flat list (`["App/Metrics","OrderValue","Service","OrdersAPI"]`), but CloudWatch expects a *list of metric definitions*, so each one must be its own nested list — hence `[["App/Metrics","OrderValue","Service","OrdersAPI"]]`. Without `region`, CloudWatch doesn't know which region to query for the metric data, so the widget renders empty even though the metric itself exists.

## Build Complete Monitoring Application

Fantastic work mastering individual CloudWatch concepts! Now it's time to bring everything together and build a complete monitoring solution from the ground up.

You have been given the basic structure of a monitoring application, but several key pieces are missing. Your objective is to implement both the metric emission and dashboard creation components to create a fully functional monitoring system.

Here's what you need to complete:

* Fill in the `put_metric_value` function with the complete metric data structure.
* Add the missing properties to make the dashboard widget display correctly.
* Implement the main monitoring loop that generates and sends sample data.

Look for the TODO comments throughout the code — they will guide you to exactly where each piece needs to be added. Remember to use consistent namespace and dimension values so your metrics and dashboard connect properly.

Once complete, your application will demonstrate the full monitoring workflow: sending custom metrics to CloudWatch and visualizing them through a dashboard that you can view in the AWS console!

```python
import time, random, json, boto3

# Initialize CloudWatch client
cw = boto3.client('cloudwatch')
NAMESPACE = "App/Metrics"
DASH_NAME = "App-Overview"

def put_metric_value(value):
    """Emit a single metric data point to CloudWatch"""
    cw.put_metric_data(
        Namespace=NAMESPACE,
        # TODO: Add the MetricData list with a dictionary containing MetricName, Dimensions, Timestamp, and Value
    )

def create_dashboard():
    """Create CloudWatch dashboard to visualize metrics"""
    widget = {
        "type":"metric","x":0,"y":0,"width":12,"height":6,
        "properties":{
            # TODO: Add the metrics array in the format [["App/Metrics","OrderValue","Service","OrdersAPI"]]
            # TODO: Add the stat property set to "Average"
            "title":"Order Value",
            # TODO: Add the region property set to "us-east-1"
            "annotations":{"horizontal":[]},
            "period":300
        }
    }
    cw.put_dashboard(
        DashboardName=DASH_NAME, 
        DashboardBody=json.dumps({"widgets":[widget]})
    )

if __name__ == "__main__":
    print("Starting application monitoring setup...")
    
    # Step 1: Emit sample metrics
    print("Emitting sample metrics...")
    # TODO: Create a for loop that runs 5 times
        # TODO: Generate a random value between 10 and 200 using random.uniform()
        # TODO: Call put_metric_value() with the generated value
        # TODO: Print the emitted metric value
        # TODO: Sleep for 1 second using time.sleep()
    
    # Step 2: Create dashboard
    print("Creating CloudWatch dashboard...")
    create_dashboard()
    print(f"Dashboard '{DASH_NAME}' created successfully!")
    
    print("Setup complete. Check AWS CloudWatch console to view your metrics and dashboard.")
```

Here is the completed monitoring application with every TODO filled in:

```python
import time, random, json, boto3

# Initialize CloudWatch client
cw = boto3.client('cloudwatch')
NAMESPACE = "App/Metrics"
DASH_NAME = "App-Overview"

def put_metric_value(value):
    """Emit a single metric data point to CloudWatch"""
    cw.put_metric_data(
        Namespace=NAMESPACE,
        MetricData=[{
            "MetricName": "OrderValue",
            "Dimensions": [{"Name":"Service","Value":"OrdersAPI"}],
            "Timestamp": time.time(),
            "Value": value
        }]
    )

def create_dashboard():
    """Create CloudWatch dashboard to visualize metrics"""
    widget = {
        "type":"metric","x":0,"y":0,"width":12,"height":6,
        "properties":{
            "metrics":[["App/Metrics","OrderValue","Service","OrdersAPI"]],
            "stat":"Average",
            "title":"Order Value",
            "region":"us-east-1",
            "annotations":{"horizontal":[]},
            "period":300
        }
    }
    cw.put_dashboard(
        DashboardName=DASH_NAME, 
        DashboardBody=json.dumps({"widgets":[widget]})
    )

if __name__ == "__main__":
    print("Starting application monitoring setup...")
    
    # Step 1: Emit sample metrics
    print("Emitting sample metrics...")
    for i in range(5):
        value = random.uniform(10, 200)
        put_metric_value(value)
        print(f"Emitted metric: OrderValue = {value:.2f}")
        time.sleep(1)
    
    # Step 2: Create dashboard
    print("Creating CloudWatch dashboard...")
    create_dashboard()
    print(f"Dashboard '{DASH_NAME}' created successfully!")
    
    print("Setup complete. Check AWS CloudWatch console to view your metrics and dashboard.")
```